# La demanda de gas y el frío

¿Cuánto gas va a pedir la Argentina en los próximos doce meses? La pregunta la contestó un
trabajo reciente para España con un SARIMA sobre 22 años de datos y un error del 5 % en el último
año, sin usar la temperatura. Acá se hace lo mismo con 30 años de datos argentinos del ENARGAS
(1996 a 2026), se suma la temperatura diaria de diez ciudades y se compara todo contra la vara
más simple: repetir el mismo mes del año anterior.

La respuesta tiene una vuelta que en España no aparece: en invierno, el consumo total argentino
no mide la demanda, mide lo que el sistema puede entregar.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent / "src"))
from datetime import date
import numpy as np, pandas as pd
from IPython.display import Image, display
from glab import DISTRIBUIDORAS, FIGURES, DATA_PROC
from glab.series import fetch, per_day, annual
from glab.clima import daily_table, weights, national, monthly, cdd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
FIGURES.mkdir(parents=True, exist_ok=True)
df = fetch()
T = daily_table()
w = weights(df[list(DISTRIBUIDORAS)])
H = national(T, w)                                   # grados-día de calefacción diarios, promedio nacional ponderado
clima = pd.DataFrame({"hdd": monthly(H), "cdd": monthly(national(T, w, cdd))})
print(f"series: {df.index.min():%m/%Y} a {df.index.max():%m/%Y}; temperatura: {T.index.min():%d/%m/%Y} a {T.index.max():%d/%m/%Y}")

## 1. Treinta años de consumo y producción

Millones de m³ por año. La producción tocó fondo en 2014, cuando el país consumía más de lo que
producía, y en 2025 volvió casi al récord de 2004 de la mano de Vaca Muerta.

In [ ]:
a = annual(df[["total", "residencial", "industria", "centrales", "gnc", "produccion"]])
a.loc[[1996, 2004, 2014, 2019, 2023, 2024, 2025]].round(0)

## 2. Dos estacionalidades opuestas

Los hogares consumen seis veces más en julio que en enero. Las centrales eléctricas tienen el pico
en verano, por el aire acondicionado, y en invierno bajan. Sumadas en el total, se tapan.

In [ ]:
rec = per_day(df[df.index.year >= 2021])
perfil = rec.groupby(rec.index.month).mean()
from glab.figuras import seasonality
seasonality(perfil, FIGURES / "estacionalidad.png"); display(Image(str(FIGURES / "estacionalidad.png")))
perfil[["residencial", "centrales", "industria", "gnc", "total"]].round(1).T

## 3. El frío no sube el total: lo redistribuye

Para cada mes de junio a agosto desde 2010, el cambio del consumo contra el mismo mes del año
anterior, al lado del cambio de los grados-día. Si el invierno es más frío, los hogares consumen
más; las centrales y la industria, menos. El total casi no se mueve.

In [ ]:
x = df.join(clima["hdd"]).dropna()
x = x[x.index.year >= 2009]
dd = (x - x.shift(12)).dropna()
inv = dd[dd.index.month.isin([6, 7, 8])]
fila = {}
for c in ["residencial", "comercial", "sdb", "industria", "gnc", "centrales", "total"]:
    b = np.polyfit(inv["hdd"], inv[c], 1)[0]
    fila[c] = {"por_grado_dia_Mm3": round(b, 2), "correlacion": round(inv[c].corr(inv["hdd"]), 2)}
from glab.figuras import redistribution
redistribution(inv, FIGURES / "redistribucion.png"); display(Image(str(FIGURES / "redistribucion.png")))
pd.DataFrame(fila).T

Cada grado-día de calefacción adicional suma unos 3 millones de m³ al mes en los hogares y
resta casi lo mismo en las centrales. El total de invierno está topeado por lo que entra al
sistema: cuando hace frío, se prioriza a los hogares y se corta a los demás. Por eso la
temperatura sirve para pronosticar a los hogares, pero no al total.

## 4. Diez años de pronósticos antes de cada invierno

Desde el fin de marzo de cada año entre 2016 y 2025, cada modelo pronostica los doce meses
siguientes con lo que sabía hasta ese momento. Los modelos con temperatura se miden dos veces:
con el tiempo que efectivamente hizo (mide el modelo) y con el promedio de los veinte años previos
(mide el pronóstico real, que no sabe si viene un invierno frío).

In [ ]:
from glab.validacion import backtest, summary
bt_total = backtest(df, clima, col="total")
bt_res = backtest(df, clima, col="residencial")
bt_total.to_csv(DATA_PROC / "backtest_total.csv", index=False); bt_res.to_csv(DATA_PROC / "backtest_residencial.csv", index=False)
s_total, s_res = summary(bt_total), summary(bt_res)
from glab.figuras import validation
validation({"Consumo total": s_total, "Consumo residencial": s_res}, FIGURES / "validacion.png"); display(Image(str(FIGURES / "validacion.png")))
print("TOTAL"); display(s_total.round(2)); print("RESIDENCIAL"); display(s_res.round(2))

En el total, el ingenuo gana: repetir el mismo mes del año pasado se equivoca 5 % en promedio,
igual que el SARIMA de España, y ninguna temperatura lo mejora. En los hogares el orden se
invierte: con el tiempo que efectivamente hace, el error baja de 11 % a 6 %. La parte que falta
para un pronóstico real es saber qué invierno viene, y eso no lo sabe nadie en marzo.

## 5. El invierno que viene, con el tiempo de cada invierno desde 1997

El modelo de los hogares se ajusta con toda la historia y se proyecta hasta septiembre de 2027 con
el tiempo real de cada año entre 1997 y 2025. No se inventa un invierno frío: se usa uno que
existió.

In [ ]:
from glab.escenarios import ensemble, winter_scenarios
e = ensemble(df["residencial"], clima)
s = winter_scenarios(e)
from glab.figuras import scenarios
scenarios(per_day(df[["residencial"]])["residencial"], per_day(s), per_day(e), FIGURES / "escenarios.png", s.attrs["anio_clima"])
display(Image(str(FIGURES / "escenarios.png")))
inv27 = per_day(e).loc["2027-06-01":"2027-08-01"].mean()
print("inviernos elegidos:", s.attrs["anio_clima"])
print(f"hogares, junio a agosto de 2027, millones de m³ por día: cálido {inv27.quantile(.1):.1f} · normal {inv27.median():.1f} · frío {inv27.quantile(.9):.1f}")
per_day(s).loc["2027-05-01":].round(1).T

## 6. Lo que queda para el notebook 02

El pronóstico mensual de los hogares funciona, pero no contesta si el gas alcanza: el problema no
está en el mes sino en el día más frío, cuando todo el sistema empuja contra el mismo caño. Eso
se mide con los partes diarios del ENARGAS en el notebook 02.

## Límites

- La temperatura es la del reanálisis ERA5 en una ciudad por distribuidora; en las áreas grandes
  (Camuzzi Sur, Gasnor) una sola ciudad es una aproximación.
- Los pesos de cada ciudad salen del consumo total de su distribuidora, no del residencial.
- Los modelos no tienen precios: los aumentos de tarifa de 2016 a 2019 aparecen como cambio de
  nivel y el SARIMA los absorbe con retraso.
- La suma de los tipos de usuario da el 94 % del total publicado; la diferencia no está abierta.